# 🎵 音符マット配置ツール — MIDI自動変換

YouTubeのURL（ピアノカバー推奨）を入力すると、
**音符マット配置ツール**にそのまま貼れる `notes` 配列を出力します。

## 使い方
1. 「ランタイム → すべてのセルを実行」を押す
2. セル③の `YOUTUBE_URL` にピアノカバー動画のURLを貼る
3. 一番下のセルを実行 → `notes:[...]` がコピー可能な形で出力される

In [ ]:
# ① 依存ライブラリのインストール（初回のみ数分かかります）
!pip install -q yt-dlp basic-pitch
!apt-get install -q ffmpeg
print('✅ インストール完了')

In [ ]:
# ② ヘルパー関数
import os, json, subprocess, numpy as np

NOTE_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

def midi_num_to_name(n):
    return f"{NOTE_NAMES[n % 12]}{n // 12 - 1}"

def transpose_to_game_range(note_name):
    """ゲームの音域 C3〜C#5 に収める"""
    if not note_name: return None
    base = note_name[:-1]
    oct_ = int(note_name[-1])
    while oct_ > 4 or (oct_ == 5 and base not in ('C', 'C#')):
        oct_ -= 1
    while oct_ < 3:
        oct_ += 1
    return f"{base}{oct_}"

def note_events_to_onpu(note_events, bpm):
    """basic-pitch の note_events → onpu-mat-tool notes配列"""
    # note_events: list of (start_sec, end_sec, pitch_midi, amplitude, bends)
    events = sorted(note_events, key=lambda x: x[0])

    # 同時発音（±50ms）でグループ化して最高音のみ残す
    WINDOW = 0.05  # 秒
    groups = []
    i = 0
    while i < len(events):
        s0 = events[i][0]
        best = events[i]
        j = i + 1
        while j < len(events) and events[j][0] - s0 <= WINDOW:
            if events[j][2] > best[2]:  # pitch_midi が大きい = 高音
                best = events[j]
            j += 1
        groups.append(best)
        i = j

    beat_sec = 60.0 / bpm
    eighth_sec = beat_sec * 0.5
    notes_out = []
    prev_end = 0.0

    for (start, end, pitch_midi, amp, bends) in groups:
        note_name = transpose_to_game_range(midi_num_to_name(int(pitch_midi)))

        # 休符
        gap = start - prev_end
        if gap > eighth_sec * 0.6:
            rd = max(0.5, round(gap / beat_sec / 0.5) * 0.5)
            if notes_out and notes_out[-1]['p'] is None:
                notes_out[-1]['d'] = round(notes_out[-1]['d'] + rd, 1)
            else:
                notes_out.append({'p': None, 'd': rd})

        dur = end - start
        d = max(0.5, round(dur / beat_sec / 0.5) * 0.5)
        notes_out.append({'p': note_name, 'd': d})
        prev_end = end

    return notes_out

def format_notes_js(notes, per_line=8):
    """notes配列をJavaScript形式に整形"""
    lines = []
    line = []
    for n in notes:
        if n['p'] is None:
            entry = f"r({n['d']})" if n['d'] != 1 else "r()"
        elif n['d'] == 1:
            entry = f"p('{n['p']}')"
        else:
            entry = f"p('{n['p']}',{n['d']})"
        line.append(entry)
        if len(line) >= per_line:
            lines.append('      ' + ','.join(line) + ',')
            line = []
    if line:
        lines.append('      ' + ','.join(line) + ',')
    return '\n'.join(lines)

print('✅ ヘルパー関数 読み込み完了')

In [ ]:
# ③ ここを編集してください
# ───────────────────────────────────────────────
YOUTUBE_URL = 'https://www.youtube.com/watch?v=XXXXXXXXX'  # ← ピアノカバーのURL
SONG_ID     = 'maribako'    # ← ツール内のID（英数字）
SONG_TITLE  = 'まり箱'       # ← 曲名
ARTIST      = '宝鐘マリン'   # ← アーティスト
CATEGORY    = 'marine'       # ← kids / marine / azki / okayu / hololive / other
BPM         = 130            # ← 曲のBPM（分からなければ120）
START_SEC   = 0              # ← 解析開始位置（秒）イントロをスキップする場合
END_SEC     = 60             # ← 解析終了位置（秒）None にすると全部
# ───────────────────────────────────────────────
print(f'設定: {SONG_TITLE} / BPM={BPM} / {START_SEC}〜{END_SEC}秒')

In [ ]:
# ④ YouTube から音声をダウンロード
import yt_dlp

WAV_PATH = f'/tmp/{SONG_ID}.wav'

ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': f'/tmp/{SONG_ID}.%(ext)s',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',
        'preferredquality': '0',
    }],
    'quiet': True,
    'no_warnings': True,
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(YOUTUBE_URL, download=True)
    print(f'✅ ダウンロード完了: {info["title"]}')
    print(f'   長さ: {info.get("duration", "?")} 秒')

In [ ]:
# ⑤ basic-pitch で音声→MIDI変換
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH
import librosa

print('🎵 音声を解析中... (1〜2分かかります)')

# 指定区間だけ切り出し
y, sr = librosa.load(WAV_PATH, sr=22050, mono=True,
                     offset=START_SEC,
                     duration=(END_SEC - START_SEC) if END_SEC else None)

# 一時WAVに書き出し
import soundfile as sf
CLIP_PATH = f'/tmp/{SONG_ID}_clip.wav'
sf.write(CLIP_PATH, y, sr)

# basic-pitch 実行
model_output, midi_data, note_events = predict(
    CLIP_PATH,
    ICASSP_2022_MODEL_PATH,
    onset_threshold=0.5,
    frame_threshold=0.3,
    minimum_note_length=58,   # 約58ms = 8分音符未満は無視
    minimum_frequency=librosa.note_to_hz('C3'),
    maximum_frequency=librosa.note_to_hz('C6'),
)

print(f'✅ 検出完了: {len(note_events)} 音符イベント')

In [ ]:
# ⑥ onpu-mat-tool 形式に変換して出力
notes = note_events_to_onpu(note_events, BPM)

# MIDIファイルも保存
MIDI_OUT = f'/tmp/{SONG_ID}.mid'
midi_data.write(MIDI_OUT)
print(f'MIDIファイル保存: {MIDI_OUT}')

# JavaScript形式で出力
js_notes = format_notes_js(notes)

print()
print('=' * 60)
print('📋 以下をコピーして songs_public.js か songs_local.js に追加:')
print('=' * 60)
output = f"""  {{
    id:'{SONG_ID}', title:'{SONG_TITLE}', artist:'{ARTIST}',
    category:'{CATEGORY}', bpm:{BPM},
    notes:[
{js_notes}
    ],
    markers:[]
  }},"""
print(output)
print('=' * 60)
print(f'総音符数: {len(notes)}')

In [ ]:
# ⑦ 【任意】生成したMIDIをColabで再生して確認
from IPython.display import Audio
import subprocess

# fluidsynth でMIDI→WAV変換して再生
!apt-get install -q fluidsynth
!fluidsynth -ni /usr/share/sounds/sf2/FluidR3_GM.sf2 {MIDI_OUT} -F /tmp/preview.wav -r 22050 2>/dev/null
Audio('/tmp/preview.wav')

In [ ]:
# ⑧ 【任意】MIDIファイルをダウンロード（ツールにドロップして使えます）
from google.colab import files
files.download(MIDI_OUT)
print(f'✅ {SONG_ID}.mid をダウンロードしました')
print('→ ツールの「♪+」ボタンにドロップすれば直接使えます')